# Boston 住宅価格予測モデルの探索と視覚化

このノートブックでは、Boston データセットを使った高度な回帰モデルの性能を視覚的に評価します。

## 目的

- **特徴量エンジニアリングを実践する** - 2 乗項・交互作用項の生成
- **データ標準化の効果を確認する** - 正規化による学習の安定化
- **回帰モデルの性能を視覚的に評価する** - 実測値 vs 予測値、残差プロット
- **特徴量の分布を理解する** - ヒストグラム、散布図、相関分析

## 1. 環境セットアップとパッケージ読み込み

In [ ]:
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: FSharp.Stats, 0.5.0"

open System
open System.IO
open Microsoft.ML
open Microsoft.ML.Data
open Plotly.NET
open Plotly.NET.LayoutObjects
open FSharp.Stats

printfn "✅ 環境セットアップ完了"

## 2. データ型定義

In [2]:
[<CLIMutable>]
type BostonData = {
    [<LoadColumn(0)>] CRIME: string
    [<LoadColumn(5)>] RM: float32
    [<LoadColumn(10)>] PTRATIO: float32
    [<LoadColumn(12)>] LSTAT: float32
    [<LoadColumn(13)>] PRICE: float32
}

[<CLIMutable>]
type BostonFeatures = {
    RM: float32
    LSTAT: float32
    PTRATIO: float32
    RM2: float32
    LSTAT2: float32
    PTRATIO2: float32
    [<ColumnName("RM_x_LSTAT")>] RMxLSTAT: float32
    PRICE: float32
}

[<CLIMutable>]
type BostonPrediction = {
    [<ColumnName("Score")>] Price: float32
}

## 3. データ読み込みと探索

In [3]:
let mlContext = MLContext(seed = Nullable 0)
let dataPath = "../data/Boston.csv"

let dataView =
    mlContext.Data.LoadFromTextFile<BostonData>(
        dataPath,
        hasHeader = true,
        separatorChar = ',')

// データフレームに変換
let bostonData =
    mlContext.Data.CreateEnumerable<BostonData>(dataView, reuseRowObject = false)
    |> Seq.toList

printfn $"データ数: {bostonData.Length} サンプル"
printfn $"\n最初の 5 件:"
bostonData
|> List.take 5
|> List.iteri (fun i d ->
    printfn $"  {i+1}. CRIME={d.CRIME}, RM={d.RM:F2}, LSTAT={d.LSTAT:F2}, PTRATIO={d.PTRATIO:F2}, PRICE=${d.PRICE:F2}k"
)

データ数: 100 サンプル

最初の 5 件:
  1. CRIME=high, RM=3.56, LSTAT=7.12, PTRATIO=20.20, PRICE=$27.50k
  2. CRIME=low, RM=5.95, LSTAT=27.71, PTRATIO=21.00, PRICE=$13.20k
  3. CRIME=very_low, RM=6.16, LSTAT=7.43, PTRATIO=14.70, PRICE=$24.10k
  4. CRIME=low, RM=6.15, LSTAT=18.46, PTRATIO=21.20, PRICE=$17.80k
  5. CRIME=high, RM=6.98, LSTAT=11.66, PTRATIO=20.20, PRICE=$29.80k


## 4. 欠損値の確認

In [4]:
let countMissing getValue name =
    let missing =
        bostonData
        |> List.filter (fun d -> Single.IsNaN(getValue d))
        |> List.length
    printfn $"{name}: {missing} 件の欠損値 ({float missing / float bostonData.Length * 100.0:F2}%%)"

printfn "\n=== 欠損値の確認 ==="
countMissing (fun d -> d.RM) "RM（平均部屋数）"
countMissing (fun d -> d.LSTAT) "LSTAT（低所得者の割合）"
countMissing (fun d -> d.PTRATIO) "PTRATIO（教員比率）"
countMissing (fun d -> d.PRICE) "PRICE（住宅価格）"


=== 欠損値の確認 ===
RM（平均部屋数）: 0 件の欠損値 (0.00%)
LSTAT（低所得者の割合）: 0 件の欠損値 (0.00%)
PTRATIO（教員比率）: 0 件の欠損値 (0.00%)
PRICE（住宅価格）: 0 件の欠損値 (0.00%)


## 5. 基本統計量

In [5]:
let printStats name getValue =
    let values =
        bostonData
        |> List.map (fun d -> float (getValue d))
        |> List.filter (fun v -> not (Double.IsNaN v))
    
    if values.Length > 0 then
        let mean = List.average values
        let std = Seq.stDev values
        let min = List.min values
        let max = List.max values
        printfn $"{name}:"
        printfn $"  平均={mean:F2}, 標準偏差={std:F2}, 最小={min:F2}, 最大={max:F2}"

printfn "\n=== 基本統計量 ==="
printStats "RM（平均部屋数）" (fun d -> d.RM)
printStats "LSTAT（低所得者の割合）" (fun d -> d.LSTAT)
printStats "PTRATIO（教員比率）" (fun d -> d.PTRATIO)
printStats "PRICE（住宅価格、$1000単位）" (fun d -> d.PRICE)

printfn "\nCRIME（犯罪率カテゴリ）分布:"
bostonData
|> List.groupBy (fun d -> d.CRIME)
|> List.iter (fun (crime, samples) -> printfn $"  {crime}: {samples.Length} サンプル")


=== 基本統計量 ===
RM（平均部屋数）:
  平均=6.24, 標準偏差=0.77, 最小=3.56, 最大=8.70
LSTAT（低所得者の割合）:
  平均=11.83, 標準偏差=6.83, 最小=1.92, 最大=30.59
PTRATIO（教員比率）:
  平均=18.52, 標準偏差=1.94, 最小=13.00, 最大=22.00
PRICE（住宅価格、$1000単位）:
  平均=23.46, 標準偏差=9.57, 最小=5.00, 最大=50.00

CRIME（犯罪率カテゴリ）分布:
  high: 25 サンプル
  low: 25 サンプル
  very_low: 50 サンプル


## 6. データ分布の視覚化

### 住宅価格の分布

In [6]:
let priceValues =
    bostonData
    |> List.map (fun d -> float d.PRICE)
    |> List.filter (fun v -> not (Double.IsNaN v))

let priceHist =
    Chart.Histogram(priceValues, Name = "住宅価格分布")
    |> Chart.withXAxisStyle(TitleText = "住宅価格（$1000単位）")
    |> Chart.withYAxisStyle(TitleText = "頻度")
    |> Chart.withTitle "Boston 住宅価格の分布"
    |> Chart.withSize(800, 500)

priceHist

<!-- Plotly chart will be drawn inside this DIV -->

### RM（平均部屋数） vs 住宅価格の散布図

In [7]:
let validData =
    bostonData
    |> List.filter (fun d -> not (Single.IsNaN d.RM) && not (Single.IsNaN d.PRICE))

let rmValues = validData |> List.map (fun d -> float d.RM)
let priceFromRM = validData |> List.map (fun d -> float d.PRICE)

let rmScatter =
    Chart.Scatter(rmValues, priceFromRM, mode = StyleParam.Mode.Markers, Name = "RM vs 価格")
    |> Chart.withMarkerStyle(Size = 8, Opacity = 0.6)
    |> Chart.withXAxisStyle(TitleText = "RM（平均部屋数）")
    |> Chart.withYAxisStyle(TitleText = "住宅価格（$1000単位）")
    |> Chart.withTitle "平均部屋数と住宅価格の関係"
    |> Chart.withSize(800, 600)

rmScatter

<!-- Plotly chart will be drawn inside this DIV -->

### LSTAT（低所得者の割合） vs 住宅価格の散布図

In [8]:
let lstatValues = validData |> List.map (fun d -> float d.LSTAT)
let priceFromLSTAT = validData |> List.map (fun d -> float d.PRICE)

let lstatScatter =
    Chart.Scatter(lstatValues, priceFromLSTAT, mode = StyleParam.Mode.Markers, Name = "LSTAT vs 価格")
    |> Chart.withMarkerStyle(Size = 8, Opacity = 0.6)
    |> Chart.withXAxisStyle(TitleText = "LSTAT（低所得者の割合 %)")
    |> Chart.withYAxisStyle(TitleText = "住宅価格（$1000単位）")
    |> Chart.withTitle "低所得者の割合と住宅価格の関係"
    |> Chart.withSize(800, 600)

lstatScatter

<!-- Plotly chart will be drawn inside this DIV -->

## 7. 特徴量エンジニアリング

元の特徴量（RM, LSTAT, PTRATIO）から、2 乗項と交互作用項を生成します。

In [9]:
// 欠損値補完（平均値で置換）
let calculateMean values =
    if List.isEmpty values then 0.0f else List.average values

let validRMs = bostonData |> List.filter (fun r -> not (Single.IsNaN(r.RM))) |> List.map (fun r -> r.RM)
let validLSTATs = bostonData |> List.filter (fun r -> not (Single.IsNaN(r.LSTAT))) |> List.map (fun r -> r.LSTAT)
let validPTRATIOs = bostonData |> List.filter (fun r -> not (Single.IsNaN(r.PTRATIO))) |> List.map (fun r -> r.PTRATIO)

let trainMean =
    Map.ofList [
        ("RM", calculateMean validRMs)
        ("LSTAT", calculateMean validLSTATs)
        ("PTRATIO", calculateMean validPTRATIOs)
    ]

let filledData =
    bostonData
    |> List.map (fun row ->
        { row with
            RM = if Single.IsNaN(row.RM) then trainMean.["RM"] else row.RM
            LSTAT = if Single.IsNaN(row.LSTAT) then trainMean.["LSTAT"] else row.LSTAT
            PTRATIO = if Single.IsNaN(row.PTRATIO) then trainMean.["PTRATIO"] else row.PTRATIO
        })

// 特徴量エンジニアリング（2乗項 + 交互作用項）
let engineeredData =
    filledData
    |> List.map (fun row ->
        {
            RM = row.RM
            LSTAT = row.LSTAT
            PTRATIO = row.PTRATIO
            RM2 = row.RM * row.RM
            LSTAT2 = row.LSTAT * row.LSTAT
            PTRATIO2 = row.PTRATIO * row.PTRATIO
            RMxLSTAT = row.RM * row.LSTAT
            PRICE = row.PRICE
        })

printfn "特徴量エンジニアリング完了"
printfn "  元の特徴量: RM, LSTAT, PTRATIO (3 個)"
printfn "  追加された特徴量: RM2, LSTAT2, PTRATIO2, RM×LSTAT (4 個)"
printfn "  合計特徴量数: 7 個"

特徴量エンジニアリング完了
  元の特徴量: RM, LSTAT, PTRATIO (3 個)
  追加された特徴量: RM2, LSTAT2, PTRATIO2, RM×LSTAT (4 個)
  合計特徴量数: 7 個


## 8. モデルの訓練と評価

In [10]:
// F# 用のダウンキャストヘルパー関数
let downcastPipeline (x: IEstimator<_>) =
    match x with
    | :? IEstimator<ITransformer> as y -> y
    | _ -> failwith "downcastPipeline: IEstimator<ITransformer> が期待されます"

let engineeredDataView = mlContext.Data.LoadFromEnumerable(engineeredData)
let trainTestSplit = mlContext.Data.TrainTestSplit(engineeredDataView, testFraction = 0.2, seed = Nullable 42)

let pipeline =
    mlContext.Transforms.Concatenate(
        "Features",
        "RM", "LSTAT", "PTRATIO", "RM2", "LSTAT2", "PTRATIO2", "RM_x_LSTAT")
    |> downcastPipeline
    |> fun estimator ->
        estimator
            .Append(mlContext.Transforms.NormalizeMinMax("Features"))
            .Append(mlContext.Regression.Trainers.Sdca(
                labelColumnName = "PRICE",
                maximumNumberOfIterations = 100))

printfn "モデルを訓練中..."
let model = pipeline.Fit(trainTestSplit.TrainSet)

// 予測
let predictions = model.Transform(trainTestSplit.TestSet)

// 評価
let metrics = mlContext.Regression.Evaluate(predictions, labelColumnName = "PRICE")

printfn "\n=== モデル評価結果 ==="
printfn $"R² (決定係数):      {metrics.RSquared:F4} ({metrics.RSquared * 100.0:F2}%%)"  
printfn $"MAE (平均絶対誤差):  ${metrics.MeanAbsoluteError:F2}k"
printfn $"RMSE (二乗平均平方根誤差): ${metrics.RootMeanSquaredError:F2}k"

モデルを訓練中...

=== モデル評価結果 ===
R² (決定係数):      0.3125 (31.25%)
MAE (平均絶対誤差):  $5.17k
RMSE (二乗平均平方根誤差): $8.67k


## 9. 実測値 vs 予測値の可視化

In [11]:
let testData =
    mlContext.Data.CreateEnumerable<BostonFeatures>(trainTestSplit.TestSet, reuseRowObject = false)
    |> Seq.toList

let predictedData =
    mlContext.Data.CreateEnumerable<BostonPrediction>(predictions, reuseRowObject = false)
    |> Seq.toList

let actualValues = testData |> List.map (fun d -> float d.PRICE)
let predictedValues = predictedData |> List.map (fun p -> float p.Price)

// 散布図
let scatterChart =
    Chart.Scatter(actualValues, predictedValues, mode = StyleParam.Mode.Markers, Name = "予測結果")
    |> Chart.withMarkerStyle(Size = 10, Opacity = 0.6)

// 理想線 (y = x)
let minVal = min (List.min actualValues) (List.min predictedValues)
let maxVal = max (List.max actualValues) (List.max predictedValues)
let idealLine =
    Chart.Line([minVal; maxVal], [minVal; maxVal], Name = "理想線 (y=x)")
    |> Chart.withLineStyle(Dash = StyleParam.DrawingStyle.Dash, Color = Color.fromKeyword Red)

let predictionChart =
    [scatterChart; idealLine]
    |> Chart.combine
    |> Chart.withXAxisStyle(TitleText = "実測値（$1000単位）")
    |> Chart.withYAxisStyle(TitleText = "予測値（$1000単位）")
    |> Chart.withTitle "実測値 vs 予測値"
    |> Chart.withSize(800, 600)

predictionChart

<!-- Plotly chart will be drawn inside this DIV -->

## 10. 残差プロット

In [12]:
let residuals =
    List.zip actualValues predictedValues
    |> List.map (fun (actual, predicted) -> actual - predicted)

let residualChart =
    Chart.Scatter(predictedValues, residuals, mode = StyleParam.Mode.Markers, Name = "残差")
    |> Chart.withMarkerStyle(Size = 8, Opacity = 0.6)
    |> Chart.withXAxisStyle(TitleText = "予測値（$1000単位）")
    |> Chart.withYAxisStyle(TitleText = "残差（実測値 - 予測値）")
    |> Chart.withTitle "残差プロット"
    |> Chart.withSize(800, 600)

residualChart

<!-- Plotly chart will be drawn inside this DIV -->

## 11. 特徴量の重要性の分析

エンジニアリングされた特徴量と住宅価格の相関を確認します。

In [13]:
printfn "\n=== 特徴量と価格の相関係数 ==="

let calculateCorrelation featureValues priceValues =
    let n = float (List.length featureValues)
    let meanFeature = List.average featureValues
    let meanPrice = List.average priceValues
    
    let numerator =
        List.zip featureValues priceValues
        |> List.sumBy (fun (f, p) -> (f - meanFeature) * (p - meanPrice))
    
    let denomFeature =
        featureValues
        |> List.sumBy (fun f -> (f - meanFeature) ** 2.0)
        |> sqrt
    
    let denomPrice =
        priceValues
        |> List.sumBy (fun p -> (p - meanPrice) ** 2.0)
        |> sqrt
    
    numerator / (denomFeature * denomPrice)

let prices = engineeredData |> List.map (fun d -> float d.PRICE)

let features = [
    ("RM", engineeredData |> List.map (fun d -> float d.RM))
    ("LSTAT", engineeredData |> List.map (fun d -> float d.LSTAT))
    ("PTRATIO", engineeredData |> List.map (fun d -> float d.PTRATIO))
    ("RM2", engineeredData |> List.map (fun d -> float d.RM2))
    ("LSTAT2", engineeredData |> List.map (fun d -> float d.LSTAT2))
    ("PTRATIO2", engineeredData |> List.map (fun d -> float d.PTRATIO2))
    ("RM×LSTAT", engineeredData |> List.map (fun d -> float d.RMxLSTAT))
]

features
|> List.iter (fun (name, values) ->
    let corr = calculateCorrelation values prices
    printfn $"  {name,-15}: {corr:F4}"
)


=== 特徴量と価格の相関係数 ===
  RM             : 0.6867
  LSTAT          : -0.6851
  PTRATIO        : -0.4538
  RM2            : 0.7335
  LSTAT2         : -0.5631
  PTRATIO2       : -0.4565
  RM×LSTAT       : -0.6682


## まとめ

この Jupyter Notebook での探索により、以下のことが明らかになりました：

- **特徴量エンジニアリングの効果** - 2 乗項と交互作用項により特徴量を 3 個から 7 個に拡張
- **データ標準化の重要性** - NormalizeMinMax により学習を安定化
- **特徴量と価格の関係** - RM（部屋数）は正の相関、LSTAT（低所得者の割合）は負の相関
- **モデルの性能** - R² 値約 31% で、さらなる改善の余地あり
- **残差の分布** - 予測誤差の傾向を視覚的に確認

### 改善案

- より多くの特徴量の追加（他のカラムの利用）
- 異なるアルゴリズムの試行（FastTree, LightGBM など）
- ハイパーパラメータのチューニング
- 外れ値の除去や変換